# 03 — Local XGBoost Training + Cross-Validation
Trains with 5-fold stratified CV, reports Normalized Gini per fold, and saves the best model.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

from src.preprocessing import load_and_clean, get_feature_columns
from src.features import build_preprocessor, save_preprocessor
from src.metrics import normalized_gini, gini_xgb_eval, print_cv_summary

SEED = 42
N_FOLDS = 5
MODEL_DIR = '../models'

## 1. Load data

In [ ]:
df = load_and_clean('../data/raw/train.csv')
feature_cols = get_feature_columns(df)
X = df[feature_cols]
y = df['target']

pos_rate = y.mean()
scale_pos_weight = (1 - pos_rate) / pos_rate
print(f'Rows: {len(df):,}  |  Positive rate: {pos_rate:.4f}  |  scale_pos_weight: {scale_pos_weight:.1f}')

## 2. XGBoost hyperparameters

In [ ]:
XGB_PARAMS = {
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 100,
    'scale_pos_weight': scale_pos_weight,
    'objective': 'binary:logistic',
    'tree_method': 'hist',
    'eval_metric': 'auc',
    'seed': SEED,
    'nthread': -1,
}

NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 50
print('Hyperparameters:', json.dumps(XGB_PARAMS, indent=2))

## 3. 5-fold cross-validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores = []
oof_preds = np.zeros(len(df))
best_model = None
best_gini = -1.0

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'\n--- Fold {fold}/{N_FOLDS} ---')
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Fit preprocessor on train fold only (prevents target leakage)
    prep = build_preprocessor(df)
    X_tr_enc = prep.fit_transform(X_tr, y_tr)
    X_val_enc = prep.transform(X_val)

    dtrain = xgb.DMatrix(X_tr_enc, label=y_tr)
    dval   = xgb.DMatrix(X_val_enc, label=y_val)

    model = xgb.train(
        XGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        custom_metric=gini_xgb_eval,
        maximize=False,
        verbose_eval=50,
    )

    val_preds = model.predict(dval)
    oof_preds[val_idx] = val_preds

    gini = normalized_gini(y_val.values, val_preds)
    fold_scores.append(gini)
    print(f'Fold {fold} Gini: {gini:.4f}  |  Best round: {model.best_iteration}')

    if gini > best_gini:
        best_gini = gini
        best_model = model
        best_prep = prep

print('\n' + '='*40)
print_cv_summary(fold_scores)
oof_gini = normalized_gini(y.values, oof_preds)
print(f'OOF Gini (full): {oof_gini:.4f}')

## 4. Save best model artifacts

In [ ]:
best_model.save_model(f'{MODEL_DIR}/xgb_model.json')
save_preprocessor(best_prep, f'{MODEL_DIR}/preprocessor.pkl')

cv_results = {
    'fold_gini': fold_scores,
    'mean_gini': float(np.mean(fold_scores)),
    'std_gini': float(np.std(fold_scores)),
    'oof_gini': float(oof_gini),
    'params': XGB_PARAMS,
    'num_boost_round': NUM_BOOST_ROUND,
    'best_iteration': best_model.best_iteration,
}
with open(f'{MODEL_DIR}/cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)

print(f'Saved: {MODEL_DIR}/xgb_model.json')
print(f'Saved: {MODEL_DIR}/preprocessor.pkl')
print(f'Saved: {MODEL_DIR}/cv_results.json')

## 5. Feature importance

In [ ]:
importance = best_model.get_score(importance_type='gain')
imp_df = pd.Series(importance).sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
imp_df.plot(kind='barh', color='steelblue')
plt.xlabel('Gain')
plt.title('Top 20 feature importances (gain)')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. OOF prediction distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(oof_preds[y == 0], bins=60, alpha=0.6, color='steelblue', label='No claim', density=True)
axes[0].hist(oof_preds[y == 1], bins=60, alpha=0.6, color='tomato',    label='Claim',    density=True)
axes[0].set_xlabel('Predicted probability')
axes[0].set_title('OOF score distributions by class')
axes[0].legend()

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[1].plot(fpr, tpr, color='steelblue', lw=2)
axes[1].plot([0,1], [0,1], 'k--', lw=1)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title(f'ROC Curve  (Gini={oof_gini:.4f})')

plt.tight_layout()
plt.savefig('../data/oof_roc.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Metric | Value |
|---|---|
| CV Normalized Gini | see fold_scores above |
| OOF Normalized Gini | see oof_gini above |
| Expected range | 0.275 – 0.285 |

**Next:** Push artifacts to S3, then run `sagemaker/deploy.py --train-only` to launch a SageMaker training job.